In [ ]:
library(glmmTMB)
library(ggplot2)
library(emmeans)
library(broom.mixed)
library(tidyverse)
library(patchwork)    
library(easystats)
library(DHARMa)
library(car)
library(multcomp)
library(knitr)
library(dplyr)
library(stringr)
library(gt)

In [ ]:
BASE_DIR <- getwd()

METHODS <- c("IDW_median_10000.tif", "NaturalNeighbor_median_10000.tif", "RBF_median_10000.tif", "TIN_median_10000.tif", "isoKrige_AIC_median_pred_10000.tif", "isoKrige_AIC_median_detrended_pred_10000.tif")
P_ADJUST <- "holm" 
ALPHA <- 0.05
SEED <- 42

RMSE_PATH <- file.path(paste0(BASE_DIR, '/analysis_data/aggregated_rmse.csv'))


CHAR_PATH <- file.path(paste0(BASE_DIR, '/analysis_data/survey_characteristics.csv'))

OUT_DIR <- file.path(paste0(BASE_DIR, "/model_outputs/figures"))

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

In [ ]:
rmse_raw <- read_csv(RMSE_PATH, show_col_types = FALSE)
chars <- read_csv(CHAR_PATH, show_col_types = FALSE)

In [ ]:
df <- rmse_raw %>%
  pivot_longer(
    cols      = all_of(METHODS),
    names_to  = "method",
    values_to = "mean_rmse"
  ) %>% 
  left_join(chars, by = "survey") %>%
  mutate(
    method = factor(method, levels = METHODS),
    survey = factor(survey)
    # block = factor(block)       # Uncomment for block CV
  ) %>%
  drop_na(mean_rmse)
df  

In [ ]:
# df$survey <- as.factor(df$survey)
# df$method <- as.factor(df$method) # Assuming method is categorical

# Check for NAs in the specific columns you are using
summary(df[c("mean_rmse", "method", "density", "depth_cv", "anisotropy", "survey")])

In [ ]:
# # 1. Create a clean, subsetted dataframe without NAs and without the extreme outliers
# df_clean <- na.omit(df[c("mean_rmse", "method", "density", "depth_cv", "anisotropy", "survey")])

# # Filter out the massive outliers (e.g., keeping only reasonable RMSE values)
# # Adjust this threshold based on what makes sense for your specific domain!
# df_clean <- subset(df_clean, mean_rmse < 10) 

# # 2. Run a simplified model without interactions first
m_safe <- glmmTMB(mean_rmse ~ method *( density + depth_cv + anisotropy) + (1 | survey), 
                  data = df, 
                  family = Gamma(link = "log"), 
                  REML = TRUE)

summary(m_safe)

# m_disp <- glmmTMB(mean_rmse ~ method *( density + depth_cv + anisotropy) + (1 | survey), 
#                   dispformula = ~ method,           
#                   data = df, 
#                   family = Gamma(link = "log"), 
#                   REML = TRUE)

# summary(m_disp)

In [ ]:
sim_res <- DHARMa::simulateResiduals(m_safe)

par(mfrow = c(2, 3))
plotQQunif(sim_res)
plotResiduals(sim_res)
plotResiduals(sim_res, form = model.frame(m_safe)$method)
plotResiduals(sim_res, form = model.frame(m_safe)$density)
plotResiduals(sim_res, form = model.frame(m_safe)$depth_cv)
plotResiduals(sim_res, form = model.frame(m_safe)$anisotropy)
par(mfrow = c(1, 1))

In [ ]:
# EMMs for each method at the mean of all covariates (i.e., z = 0)
emm_methods <- emmeans(m_safe, ~ method, type = "response")
# summary(emm_methods)
plot(emm_methods, comparisons = TRUE) + 
  labs(x = "Predicted RMSE", y = NULL)

In [ ]:
emmip(m_safe, method ~ density,
      at = list(density = seq(-2, 2, by = 0.5)),
      type = "response", CIs = TRUE) +
  labs(y = "Predicted RMSE", x = "Sampling Density (z)")

emmip(m_safe, method ~ depth_cv,
      at = list(depth_cv = seq(-2, 2, by = 0.5)),
      type = "response", CIs = TRUE) +
  labs(y = "Predicted RMSE", x = "Depth CV (z)")

emmip(m_safe, method ~ anisotropy,
      at = list(anisotropy = seq(-2, 2, by = 0.5)),
      type = "response", CIs = TRUE) +
  labs(y = "Predicted RMSE", x = "Sampling Anisotropy (z)")

In [ ]:
z_levels <- c(-1, 0, 1)

density_tbl <- cld(
  emmeans(
    m_safe,
    ~ method | density,
    at = list(density = z_levels),
    type = "response"
  ),
  Letters = letters,
  adjust = "sidak"
) |>
  as.data.frame() |>
  mutate(
    .group = str_squish(.group),
    density = factor(density, levels = z_levels)
  ) |>
  dplyr::select(density, method, response, SE, asymp.LCL, asymp.UCL, .group) |>
  arrange(density, response)

density_kable <- density_tbl |>
  knitr::kable(
    col.names = c("Density", "Method", "Estimated RMSE", "SE", "Lower CI", "Upper CI", "Letters"),
    digits = 3,
    caption = "Method ranking by density"
  )

density_kable

In [ ]:
depth_cv_tbl <- cld(
  emmeans(
    m_safe,
    ~ method | depth_cv,
    at = list(depth_cv = z_levels),
    type = "response"
  ),
  Letters = letters,
  adjust = "sidak"
) |>
  as.data.frame() |>
  mutate(
    .group = str_squish(.group),
    depth_cv = factor(depth_cv, levels = z_levels)
  ) |>
  dplyr::select(depth_cv, method, response, SE, asymp.LCL, asymp.UCL, .group) |>
  arrange(depth_cv, response)

depth_cv_kable <- depth_cv_tbl |>
  knitr::kable(
    col.names = c("depth_cv", "Method", "Estimated RMSE", "SE", "Lower CI", "Upper CI", "Letters"),
    digits = 3,
    caption = "Method ranking by depth_cv"
  )

depth_cv_kable

In [ ]:
anisotropy_tbl <- cld(
  emmeans(
    m_safe,
    ~ method | anisotropy,
    at = list(anisotropy = z_levels),
    type = "response"
  ),
  Letters = letters,
  adjust = "sidak"
) |>
  as.data.frame() |>
  mutate(
    .group = str_squish(.group),
    anisotropy = factor(anisotropy, levels = z_levels)
  ) |>
  dplyr::select(anisotropy, method, response, SE, asymp.LCL, asymp.UCL, .group) |>
  arrange(anisotropy, response)

anisotropy_kable <- anisotropy_tbl |>
  knitr::kable(
    col.names = c("anisotropy", "Method", "Estimated RMSE", "SE", "Lower CI", "Upper CI", "Letters"),
    digits = 3,
    caption = "Method ranking by anisotropy"
  )

anisotropy_kable

In [ ]:
pairwise_tbl <- pairs(emm_methods, adjust = "tukey") |>
  as.data.frame() |>
  mutate(
    contrast = str_replace_all(contrast, "_median_10000\\.tif", ""),
    sig = case_when(
      p.value < 0.001 ~ "***",
      p.value < 0.01  ~ "**",
      p.value < 0.05  ~ "*",
      TRUE            ~ ""
    ),
    p_value_fmt = if_else(p.value < 1e-4, "<0.0001", sprintf("%.4f", p.value))
  ) |>
  arrange(p.value) |>
  dplyr::select(contrast, ratio, SE, z.ratio, p_value_fmt, sig)

pairwise_kable <- pairwise_tbl |>
  knitr::kable(
    col.names = c("Contrast", "Ratio", "SE", "z-ratio", "Adjusted p", "Sig"),
    digits = 3,
    caption = "Pairwise method comparisons (Tukey-adjusted)"
  )

pairwise_kable

In [ ]:
# Compare methods at low, mean, and high density

emm_by_density <- emmeans(
  m_safe,
  pairwise ~ method | density,
  at = list(density = c(-1, 0, 1)),
  type = "response"
)

pairwise_by_density_tbl <- summary(
  emm_by_density$contrasts,
  adjust = "tukey"
) |>
  as.data.frame() |>
  mutate(
    density = factor(density, levels = c(-1, 0, 1), labels = c("Low (-1)", "Mean (0)", "High (1)")),
    contrast = str_replace_all(contrast, "_median_10000\\.tif", ""),
    contrast = str_replace_all(contrast, "isoKrige_AIC_median_pred", "isoKrige_AIC"),
    contrast = str_replace_all(contrast, "isoKrige_AIC_median_detrended_pred", "isoKrige_AIC_detrended"),
    sig = case_when(
      p.value < 0.001 ~ "***",
      p.value < 0.01  ~ "**",
      p.value < 0.05  ~ "*",
      TRUE            ~ ""
    ),
    p_adj = if_else(p.value < 1e-4, "<0.0001", sprintf("%.4f", p.value))
  ) |>
  arrange(density, p.value) |>
  dplyr::select(density, contrast, ratio, SE, z.ratio, p_adj, sig)

pairwise_by_density_kable <- pairwise_by_density_tbl |>
  knitr::kable(
    col.names = c("Density", "Contrast", "Ratio", "SE", "z-ratio", "Adj p", "Sig"),
    digits = 3,
    caption = "Pairwise method comparisons by density (Tukey-adjusted)"
  )

pairwise_by_density_kable

In [ ]:
# Compare methods at low, mean, and high variation
emm_by_cv <- emmeans(
  m_safe,
  pairwise ~ method | depth_cv,
  at = list(depth_cv = c(-1, 0, 1)),
  type = "response"
)

pairwise_by_depth_cv_tbl <- summary(
  emm_by_cv$contrasts,
  adjust = "tukey"
) |>
  as.data.frame() |>
  mutate(
    depth_cv = factor(depth_cv, levels = c(-1, 0, 1), labels = c("Low (-1)", "Mean (0)", "High (1)")),
    contrast = str_replace_all(contrast, "_median_10000\\.tif", ""),
    contrast = str_replace_all(contrast, "isoKrige_AIC_median_pred", "isoKrige_AIC"),
    contrast = str_replace_all(contrast, "isoKrige_AIC_median_detrended_pred", "isoKrige_AIC_detrended"),
    sig = case_when(
      p.value < 0.001 ~ "***",
      p.value < 0.01  ~ "**",
      p.value < 0.05  ~ "*",
      TRUE            ~ ""
    ),
    p_adj = if_else(p.value < 1e-4, "<0.0001", sprintf("%.4f", p.value))
  ) |>
  arrange(depth_cv, p.value) |>
  dplyr::select(depth_cv, contrast, ratio, SE, z.ratio, p_adj, sig)

pairwise_by_depth_cv_kable <- pairwise_by_depth_cv_tbl |>
  knitr::kable(
    col.names = c("depth_cv", "Contrast", "Ratio", "SE", "z-ratio", "Adj p", "Sig"),
    digits = 3,
    caption = "Pairwise method comparisons by depth_cv (Tukey-adjusted)"
  )

pairwise_by_depth_cv_kable

In [ ]:
# Compare methods at low, mean, and high sampling anisotropy
emm_by_aniso <- emmeans(
  m_safe,
  pairwise ~ method | anisotropy,
  at = list(anisotropy = c(-1, 0, 1)),
  type = "response"
)

pairwise_by_anisotropy_tbl <- summary(
  emm_by_aniso$contrasts,
  adjust = "tukey"
) |>
  as.data.frame() |>
  mutate(
    anisotropy = factor(anisotropy, levels = c(-1, 0, 1), labels = c("Low (-1)", "Mean (0)", "High (1)")),
    contrast = str_replace_all(contrast, "_median_10000\\.tif", ""),
    contrast = str_replace_all(contrast, "isoKrige_AIC_median_pred", "isoKrige_AIC"),
    contrast = str_replace_all(contrast, "isoKrige_AIC_median_detrended_pred", "isoKrige_AIC_detrended"),
    sig = case_when(
      p.value < 0.001 ~ "***",
      p.value < 0.01  ~ "**",
      p.value < 0.05  ~ "*",
      TRUE            ~ ""
    ),
    p_adj = if_else(p.value < 1e-4, "<0.0001", sprintf("%.4f", p.value))
  ) |>
  arrange(anisotropy, p.value) |>
  dplyr::select(anisotropy, contrast, ratio, SE, z.ratio, p_adj, sig)

pairwise_by_anisotropy_kable <- pairwise_by_anisotropy_tbl |>
  knitr::kable(
    col.names = c("anisotropy", "Contrast", "Ratio", "SE", "z-ratio", "Adj p", "Sig"),
    digits = 3,
    caption = "Pairwise method comparisons by anisotropy (Tukey-adjusted)"
  )

pairwise_by_anisotropy_kable